# Trabajo Integrador
## Introducción al Análisis de Datos - UTN
## Prof: Cintia Rigoni
## Alumnos:
- Mercado Leandro
- Ramirez Rodrigo

### Fase de Limpieza, Transformación y Feature Engineering

Pre-procesamiento de Datos Crudos

- Normalización de Formatos: Eliminación manual de prefijos/sufijos no numéricos y signos negativos para estandarizar valores absolutos.  


In [5]:
import pandas as pd

df_dirt = pd.read_csv("data/dataset.csv")

df_dirt.info()

print(f"Total de registros duplicados {df_dirt.duplicated().sum()}")
print(f"Total de registros nulos {df_dirt.isnull().sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID_Transaccion   15000 non-null  object 
 1   Fecha            15000 non-null  object 
 2   ID_Cliente       15000 non-null  int64  
 3   Pais             15000 non-null  object 
 4   Ciudad           14400 non-null  object 
 5   Categoria        14700 non-null  object 
 6   Subcategoria     15000 non-null  object 
 7   Producto         15000 non-null  object 
 8   Cantidad         14551 non-null  float64
 9   Precio_Unitario  14551 non-null  float64
 10  Costo_Unitario   14550 non-null  float64
 11  Venta_Total      15000 non-null  float64
 12  Costo_Total      15000 non-null  float64
 13  Metodo_Pago      14400 non-null  object 
dtypes: float64(5), int64(1), object(8)
memory usage: 1.6+ MB
Total de registros duplicados 0
Total de registros nulos ID_Transaccion       0
Fecha        

## Procesamiento Avanzado (Python/Pandas)

- Carga de datos con gestión dinámica de tipos para asegurar el formato float64 en métricas financieras y datetime64 para el eje temporal.  

- Normalización de Texto: Aplicación de métodos .str.strip().str.title() para eliminar espacios residuales y uniformar categorías, evitando duplicidad de etiquetas en visualizaciones.  

In [6]:
# Copia de dataframe original para mantener integridad de datos originales
df_clean = df_dirt.copy()

# Columnas numericas para verificar valores invalidos
columnas_num = ['Cantidad', 'Precio_Unitario', 'Costo_Unitario', 'Venta_Total', 'Costo_Total']


for col in columnas_num:
    # transformacion a NaN
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Eliminacion de datos sin precio costo o cantidad
df_clean.dropna(subset=columnas_num, inplace=True)

# Normalizacion de columnas de texto
col_text = ["Pais", "Ciudad", "Categoria", "Subcategoria", "Producto", "Metodo_Pago"]

for col in col_text:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

### Transformación y Feature Engineering

- Imputación y Limpieza: Se aplicó fillna con etiquetas de control ("Sin Especificar", "Otros") para preservar la integridad del dataset y se eliminaron registros duplicados.  

- Cálculos Vectoriales: Se recalcularon Venta_Total, Costo_Total y Ganancia_Bruta para asegurar la coherencia matemática en el 100% de los registros.  

- Precisión Financiera: Se redondeó toda métrica monetaria a 2 decimales para corregir errores de punto flotante y estandarizar la visualización.  

- Normalización Temporal: Conversión a datetime y creación de la columna mes_texto (formato "01-Enero") para garantizar un orden cronológico correcto en los reportes.

In [7]:
fill_values = {
    'Ciudad': 'Sin Especificar',
    'Metodo_Pago': 'No Identificado',
    'Categoria': 'Otros'
}
df_clean.fillna(value=fill_values, inplace=True)

#Eliminacion de duplicados
df_clean.drop_duplicates(inplace=True)

# Recalculamos la Venta Total
df_clean['Venta_Total'] = df_clean['Precio_Unitario'] * df_clean['Cantidad']

# Recalculamos el Costo Total
df_clean['Costo_Total'] = df_clean['Costo_Unitario'] * df_clean['Cantidad']
# Creacion de columna Ganancia_Bruta

df_clean["Ganancia_Bruta"] = df_clean["Venta_Total"] - df_clean["Costo_Total"]

columnas_num.append("Ganancia_Bruta")

for col in columnas_num:
    df_clean[col] = df_clean[col].round(2)

# 2. Crear la columna mes_text
df_clean['Fecha'] = pd.to_datetime(df_clean['Fecha'])
df_clean['mes_texto'] = df_clean['Fecha'].dt.month_name()

# Conversion manual a Español y numero de mes
meses_es = {
    'January': '01-Enero', 'February': '02-Febrero', 'March': '03-Marzo',
    'April': '04-Abril', 'May': '05-Mayo', 'June': '06-Junio',
    'July': '07-Julio', 'August': '08-Agosto', 'September': '09-Septiembre',
    'October': '10-Octubre', 'November': '11-Noviembre', 'December': '12-Diciembre'
}

df_clean['mes_texto'] = df_clean['mes_texto'].map(meses_es)

### Tratamiento de Outliers y Segmentación de Clientes
  

- Filtro Estadístico (IQR): Aplicación del método de Rango Intercuartílico para eliminar registros atípicos en Venta_Total, asegurando que los promedios y métricas finales no presenten sesgos por valores extremos.  

- Análisis de Recurrencia: Generación de la métrica Total_Compras mediante un mapeo de frecuencias por ID_Cliente, permitiendo cuantificar el nivel de actividad de cada usuario.  

- Segmentación Estratégica: Creación del Índice de Constancia mediante una función personalizada (01-Fiel, 02-Regular, 03-Ocasional), categorizando a los clientes según su volumen de transacciones para análisis específicos de fidelidad.  

- Exportación de Datos: Validación final de integridad (cero nulos y duplicados) y guardado del dataset procesado en formato CSV para la fase de visualización.

In [8]:
# Eliminacion de Outliers
# 1. Calculamos los cuartiles usando la función quantile() de Pandas
q1 = df_clean['Venta_Total'].quantile(0.25)
q3 = df_clean['Venta_Total'].quantile(0.75)

# 2. Calculamos el IQR y los límites
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr # Restamos al cuartil 1
upper_limit = q3 + 1.5 * iqr # Sumamos al cuartil 3

# 3. Aplicamos la máscara booleana para QUEDARNOS solo con los datos normales
normal_mask = (df_clean['Venta_Total'] >= lower_limit) & (df_clean['Venta_Total'] <= upper_limit)
df_final = df_clean[normal_mask].copy()

# Verificamos cuántos datos se filtraron
print(f"Datos originales: {len(df_clean)}")
print(f"Datos tras quitar outliers: {len(df_final)}")

df_clean = df_final

# Creacion de diccionario con conteo de total de compras de cada cliente
dict_count = df_clean["ID_Cliente"].value_counts().to_dict()
df_clean['Total_Compras'] = df_clean['ID_Cliente'].map(dict_count)
print(dict_count)

def segment_client(cantidad):
    if cantidad >= 10:
        return '01-Fiel (VIP)'
    elif cantidad >= 4:
        return '02-Regular'
    return '03-Ocasional'

df_clean['Indice_Constancia'] = df_clean['Total_Compras'].apply(segment_client)


# Muestra en pantalla de resultados
print(f"Total de registros duplicados {df_clean.duplicated().sum()}")
print(f"Total de registros nulos {df_clean.isnull().sum()}")
df_clean.info()

display(df_clean.head(5))

# Guardado en csv
df_clean.to_csv("data/dataset_final.csv", index=False)

Datos originales: 13681
Datos tras quitar outliers: 12234
{1617: 19, 1686: 16, 795: 15, 345: 14, 1669: 14, 1127: 13, 1643: 13, 1952: 13, 1351: 13, 810: 13, 809: 13, 457: 13, 1476: 13, 1665: 13, 1674: 13, 1226: 13, 1886: 13, 141: 12, 824: 12, 878: 12, 1912: 12, 988: 12, 1133: 12, 1439: 12, 899: 12, 1083: 12, 385: 12, 624: 12, 588: 12, 1419: 12, 1089: 12, 37: 12, 1018: 12, 1383: 12, 245: 12, 174: 12, 1289: 12, 578: 12, 1281: 12, 459: 12, 13: 12, 307: 12, 499: 12, 844: 12, 324: 12, 1402: 12, 849: 12, 208: 12, 1599: 11, 217: 11, 1700: 11, 1793: 11, 1995: 11, 1532: 11, 1968: 11, 1929: 11, 462: 11, 1904: 11, 1280: 11, 382: 11, 994: 11, 1853: 11, 614: 11, 968: 11, 1478: 11, 1117: 11, 1168: 11, 716: 11, 1148: 11, 1824: 11, 365: 11, 1999: 11, 1114: 11, 1827: 11, 555: 11, 552: 11, 1345: 11, 285: 11, 1384: 11, 1537: 11, 1619: 11, 284: 11, 118: 11, 1587: 11, 185: 11, 522: 11, 1707: 11, 825: 11, 1394: 10, 183: 10, 622: 10, 1876: 10, 605: 10, 406: 10, 1071: 10, 1773: 10, 484: 10, 1837: 10, 319: 10, 

,ID_Transaccion,Fecha,ID_Cliente,Pais,Ciudad,Categoria,Subcategoria,Producto,Cantidad,Precio_Unitario,Costo_Unitario,Venta_Total,Costo_Total,Metodo_Pago,Ganancia_Bruta,mes_texto,Total_Compras,Indice_Constancia
0,TRX-105182,2023-01-01,665,Argentina,Buenos Aires,Electrónica,Laptops,Prod_Laptops_5,4.0,123.04,75.99,492.16,303.96,Transferencia,188.20,01-Enero,6,02-Regular
1,TRX-112033,2023-01-01,1246,Perú,Lima,Hogar,Muebles,Prod_Muebles_17,3.0,58.36,44.50,175.08,133.50,Crédito,41.58,01-Enero,10,01-Fiel (VIP)
3,TRX-108326,2023-01-01,1587,Argentina,Buenos Aires,Ropa,Infantil,Prod_Infantil_1,2.0,22.25,12.15,44.50,24.30,Crédito,20.20,01-Enero,11,01-Fiel (VIP)
4,TRX-104060,2023-01-01,1672,Chile,Santiago,Electrónica,Tablets,Prod_Tablets_6,4.0,132.67,68.71,530.68,274.84,Efectivo,255.84,01-Enero,7,02-Regular
5,TRX-109549,2023-01-01,124,Argentina,Córdoba,Ropa,Dama,Prod_Dama_9,7.0,33.91,17.40,237.37,121.80,Transferencia,115.57,01-Enero,8,02-Regular
